In [10]:
import pandas as pd
import string
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
nlp = spacy.load("en_core_web_sm", exclude=["parser", "ner"])

In [12]:
import json

try:
    # Read the file line by line and parse each line as a JSON object
    data = []
    with open('/content/News_Category_Dataset_v3.json', 'r') as f:
        for line in f:
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Skipping malformed JSON line: {line.strip()} - Error: {e}")
    df = pd.DataFrame(data)

except FileNotFoundError:
    print("Error: Please download the dataset from Kaggle and place it in the directory.")
    exit()

Skipping malformed JSON line: {"category": "ENTERTAINMENT", "headline": "'You Raped Women, Bill Cosby': How A Bit In Hannibal Buress's Standup Led To The Takedown Of An Icon", "authors": "Suzy Byrne, Yahoo Entertainment", - Error: Expecting property name enclosed in double quotes: line 1 column 193 (char 192)


In [13]:
# Random subset of 10,000 headlines
subset_df = df.sample(n=10000, random_state=42).reset_index(drop=True)
headlines = subset_df['headline'].tolist()

def preprocess_query(text):
    doc = nlp(text)
    processed_tokens = [
        str(token.lemma_).lower()
        for token in doc
        if not token.is_stop and not token.is_punct
    ]
    return " ".join(processed_tokens)

In [14]:
print("Preprocessing text in batch")
preprocessed_docs = []

# Sfruttiamo nlp.pipe per efficienza sui 10.000 documenti
for doc in nlp.pipe(headlines, batch_size=256):
    processed_tokens = [
        str(token.lemma_).lower()
        for token in doc
        if not token.is_stop and not token.is_punct
    ]
    preprocessed_docs.append(" ".join(processed_tokens))

#Vector Space Model Representation
vectorizer = TfidfVectorizer()

# Convertiamo i documenti preprocessati in TF-IDF matrix
tfidf_matrix = vectorizer.fit_transform(preprocessed_docs)

# Salviamo il vocabolario sul vectorizer
print(f"Vocabulary size: {len(vectorizer.vocabulary_)} words")

Preprocessing text in batch
Vocabulary size: 13125 words


In [15]:
def retrieve_documents(query, vectorizer, tfidf_matrix, original_docs, top_n=5):
    preprocessed_query = preprocess_query(query)

    # Convert TF-IDF vector
    query_vector = vectorizer.transform([preprocessed_query])

    # calcolo della cosine similarity tra la query e i documenti
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()

    # Indici di similarità della top N valori
    top_indices = similarities.argsort()[-top_n:][::-1]

    print(f"\nTop {top_n} Results for: '{query}'")
    for i, idx in enumerate(top_indices):
        score = similarities[idx]
        print(f"{i+1}. [Score: {score:.4f}] {original_docs[idx]}")

    return top_indices

if __name__ == "__main__":
    print("\nDocument Retrieval System Ready")
    while True:
        user_query = input("\nEnter a query (or type 'exit' to quit): ")
        if user_query.lower() == 'exit':
            break
        if not user_query.strip():
            continue

        retrieve_documents(user_query, vectorizer, tfidf_matrix, headlines)


Document Retrieval System Ready

Enter a query (or type 'exit' to quit): basketball championship finals

Top 5 Results for: 'basketball championship finals'
1. [Score: 0.2773] British Woman Conquers World Conkers Championship With Only 20 Minutes Of Training
2. [Score: 0.2335] Florida Basketball Player Who Collapsed During Game Is Released From Hospital
3. [Score: 0.2187] Kobe Bryant One Of 8 Finalists For Basketball Hall Of Fame
4. [Score: 0.2165] John Lewis’s Final Fight For Voting Rights
5. [Score: 0.2089] Virginia State University President Takes Student Back To School In Viral Basketball Clip

Enter a query (or type 'exit' to quit): government budget deficit

Top 5 Results for: 'government budget deficit'
1. [Score: 0.3063] Trump Hits A New High-Water Mark: The Biggest Federal Deficit In 6 Years
2. [Score: 0.2933] GOP No Longer Cares About The Deficit If It Means Women Having More Babies
3. [Score: 0.2574] Democrats Consider Abandoning Budget Floor Vote
4. [Score: 0.2488] Congres

# Query di esempio:
- "government budget deficit" (Politics)
- "apple smartphone release" (Tech)
- "basketball championship finals" (Sports)
- "climate change emissions" (Environment)
- "oscar awards red carpet" (Entertainment)
- "stock market crash" (Business)
- "healthy diet recipes" (Wellness/Food)
- "supreme court ruling" (Politics/U.S. News)
- "artificial intelligence breakthrough" (Tech/Science)
- "travel restrictions europe" (Travel)

